# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 11 · Direct-state and goal-geometry ablation

**One fixed first fold; four new coordinate fits; no algorithm search or ensemble.**

Control: the preserved 72-column Round 3 model. Treatment 1 adds 62 direct observed-state fields. Treatment 2 adds 26 goal-frame descriptors/support flags on top of treatment 1. The candidate schemas have 72, 134 and 160 columns before training-only screening.

This remains an exploratory study on the same reused games. A passing feature gate earns consideration for broader validation; it is not a Kaggle score or model-release approval.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round4')
OUT = Path('/home/sagemaker-user/nfl-feature-round4-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open this kit in the existing NFL space with its existing Python environment.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'

def run(stage):
    command = [str(PY), str(KIT / 'run_round.py'), stage, '--out', str(OUT)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        process.wait(timeout=10)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped (exit {code}). Preserve checkpoints and export the report. Do not change settings or reinstall packages.')

def show(fig, name):
    visuals.save(fig, OUT, name).show()


## 1 · Verify readiness

Complete notebook 10 first. Old control predictions will be checked again before fitting. No raw labels are regenerated. The estimator, seed, training exposure and row order stay fixed.

In [ ]:
prepared = visuals.read(OUT / 'preparation.json')
assert prepared['status'] == 'state_features_ready'
print('Prepared plays:', prepared['plays'])
print('New feature columns:', prepared['direct_state_columns'] + prepared['goal_columns'])


## 2 · Run the bounded first-fold comparison

Maximum: four new coordinate estimators, 120 boosting iterations each, checkpointed every 30 iterations. The control models are never fitted here. The hard cap is 240 seconds and the runner uses two CPU threads. Time limits are safeguards, not promised runtimes.

Retain a failed feature comparison as evidence. Do not tune its hyperparameters or introduce a role-gated combination after inspecting it.

In [ ]:
run('fit')
result = visuals.read(OUT / 'summary.json')
assert result['status'] == 'round4_first_fold_complete'
print(json.dumps({k: result[k] for k in ('pooled_rmse', 'screen_passed_arms', 'new_coordinate_models', 'parent_refits')}, indent=2))


## 3 · Attribute the result

Compare direct state against the preserved control, then geometry against direct state. Geometry must also beat the preserved control. The exploratory rule requires at least 1% lower RMSE and a negative adjusted upper paired-game difference bound. The six-look adjustment does not erase repeated use of these games across prior rounds.

Role and horizon charts are diagnostics only—not an invitation to remove hard rows or select favorable subsets.

In [ ]:
show(visuals.scores(result, 'Round 4 · Matched first-fold coordinate RMSE'), 'round4_scores')
show(visuals.contrasts(result, 'Round 4 · Planned paired comparisons'), 'round4_contrasts')
show(visuals.horizon_errors(result), 'round4_horizon_errors')
show(visuals.role_errors(result), 'round4_role_errors')


## 4 · Verify fresh-process numerical replay

This command reloads the completed new models and the preserved control. It must not create a missing model or resume incomplete training. All new predictions, row keys and targets must exactly match their saved artifacts.

In [ ]:
run('replay')
replay = visuals.read(OUT / 'replay.json')
assert replay['new_coordinate_models'] == 0
assert replay['parent_refits'] == 0
assert replay['all_new_prediction_replays_exact']
assert replay['pooled_rmse'] == result['pooled_rmse']
print('Fresh-process replay passed; no models were refitted.')


## 5 · Report and stop

Save the notebook and download the report ZIP below. Return it before running any broader experiment, including a second fold. No GitHub update, new Kaggle score or completed feature-research claim is implied.

If no feature passes, stop these two treatments unchanged. Learned temporal/player representations and other high-value avenues remain open; the goal is not an endless sequence of arbitrary tree-column screens.

In [ ]:
run('report')
print('Download:', OUT / 'nfl_feature_round4_report.zip')
print('Save notebooks, then stop the existing space. Do not delete it.')
